# GateSense 01: train the number-plate detector (YOLOv8n vs YOLOv8s)

Run this in **Google Colab with a GPU**: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

It downloads a public Indian number-plate dataset (or uses your own photos), fine-tunes **YOLOv8n** and **YOLOv8s**, measures them on held-out images, picks one **by the numbers**, exports it to **ONNX** and gives you files to download.

**Before you run (once):** the default dataset comes from Roboflow Universe, which needs a free API key.
1. Sign up at roboflow.com (free), open *Settings → API Keys*, copy your **private API key**.
2. In Colab, click the **key icon (Secrets)** in the left sidebar → *Add new secret* → name `ROBOFLOW_API_KEY`, paste the key, and switch on *Notebook access*.

Never paste the key into a cell or into a chat. To use your own photos instead, set `DATASET_SOURCE = "drive"` below (see `anpr/DATASET.md`).

Takes about 30–60 minutes on a T4, depending on the dataset size and `EPOCHS`.

In [ ]:
# --- Settings (the defaults work; change only if you know why) ---
DATASET_SOURCE = "roboflow"  # "roboflow" or "drive"
# A public CC BY 4.0 Indian plate dataset on Roboflow Universe (any YOLO-format plate set works):
ROBOFLOW_WORKSPACE = "licenceplatedetection-i880o"
ROBOFLOW_PROJECT = "indian-plate-detection-object-detection"
ROBOFLOW_VERSION = None  # None = latest version
# Used when DATASET_SOURCE == "drive": a folder with data.yaml + train/valid(/test) in YOLO format
DRIVE_DATASET = "/content/drive/MyDrive/gatesense/detector_dataset"

CANDIDATES = ["yolov8n.pt", "yolov8s.pt"]  # pretrained on COCO, fine-tuned here
EPOCHS = 60
IMAGE_SIZE = 640
# Pick the bigger model only if it is clearly more accurate (mAP50-95 points).
MIN_GAIN_FOR_BIGGER_MODEL = 0.02

In [ ]:
import subprocess, sys
print(sys.version)
assert sys.version_info >= (3, 10), "Colab should run Python 3.10+"
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU: Runtime -> Change runtime type -> T4 GPU (training on CPU takes hours)")
%pip install -q "ultralytics>=8.3" roboflow onnx onnxslim onnxruntime

In [ ]:
import os, shutil, yaml
from pathlib import Path

if DATASET_SOURCE == "roboflow":
    from google.colab import userdata
    from roboflow import Roboflow

    api_key = userdata.get("ROBOFLOW_API_KEY")  # from the Secrets panel, never typed here
    project = Roboflow(api_key=api_key).workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = ROBOFLOW_VERSION
    if version is None:
        version = max(int(str(v.version).rstrip("/").split("/")[-1]) for v in project.versions())
    print("dataset version", version)
    dataset_dir = Path(project.version(version).download("yolov8", location="/content/dataset").location)
elif DATASET_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    dataset_dir = Path("/content/dataset")
    shutil.copytree(DRIVE_DATASET, dataset_dir, dirs_exist_ok=True)
else:
    raise ValueError("DATASET_SOURCE must be 'roboflow' or 'drive'")
print(sorted(p.name for p in dataset_dir.iterdir()))

### Make it a clean one-class dataset
Public datasets name the class differently (`plate`, `number plate`, `License_Plate`…) and some have extra classes. We map every box to one class, `plate`, and write absolute paths into `data.yaml` (Roboflow's relative paths often break).

In [ ]:
splits = {name: dataset_dir / name / "images" for name in ("train", "valid", "test")}
splits = {name: path for name, path in splits.items() if path.exists()}
assert "train" in splits and "valid" in splits, f"expected train/ and valid/ folders, got {list(splits)}"

boxes = 0
for split in splits:
    for label_file in (dataset_dir / split / "labels").glob("*.txt"):
        lines = [l.split() for l in label_file.read_text().splitlines() if l.strip()]
        label_file.write_text("".join("0 " + " ".join(parts[1:5]) + "\n" for parts in lines if len(parts) >= 5))
        boxes += len(lines)

data_yaml = dataset_dir / "gatesense_plate.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(dataset_dir), "train": "train/images", "val": "valid/images",
    **({"test": "test/images"} if "test" in splits else {}),
    "nc": 1, "names": ["plate"],
}))
counts = {split: len(list(path.glob("*"))) for split, path in splits.items()}
print("images per split:", counts, "| plate boxes:", boxes)

In [ ]:
import json, time
from ultralytics import YOLO

EVAL_SPLIT = "test" if "test" in splits else "val"  # never score on the images we trained on
results = []
for weights in CANDIDATES:
    name = Path(weights).stem + "_plate"
    started = time.time()
    model = YOLO(weights)
    model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMAGE_SIZE, patience=15, seed=0,
                project="/content/runs", name=name, exist_ok=True, verbose=False)
    best = YOLO(f"/content/runs/{name}/weights/best.pt")
    metrics = best.val(data=str(data_yaml), split=EVAL_SPLIT, imgsz=IMAGE_SIZE, verbose=False)
    onnx_path = best.export(format="onnx", imgsz=IMAGE_SIZE, opset=17, simplify=True)
    results.append({
        "model": name, "eval_split": EVAL_SPLIT, "eval_images": counts.get("test" if EVAL_SPLIT == "test" else "valid"),
        "map50": round(float(metrics.box.map50), 4), "map50_95": round(float(metrics.box.map), 4),
        "precision": round(float(metrics.box.mp), 4), "recall": round(float(metrics.box.mr), 4),
        "train_minutes": round((time.time() - started) / 60, 1), "epochs": EPOCHS,
        "onnx": str(onnx_path), "onnx_mb": round(Path(onnx_path).stat().st_size / 1e6, 1),
    })
    print(results[-1])

In [ ]:
import hashlib, platform

small, big = results[0], results[-1]
chosen = big if big["map50_95"] - small["map50_95"] >= MIN_GAIN_FOR_BIGGER_MODEL else small
reason = (f"{chosen['model']}: " + ("clearly more accurate" if chosen is big and big is not small
          else f"the larger model gains < {MIN_GAIN_FOR_BIGGER_MODEL:.2f} mAP50-95, and the small one is ~3x faster on CPU"))

out = Path("/content/gatesense_results"); out.mkdir(exist_ok=True)
shutil.copy(chosen["onnx"], out / "plate_detector.onnx")
sha = hashlib.sha256((out / "plate_detector.onnx").read_bytes()).hexdigest()
source = (f"Roboflow `{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}` v{version}" if DATASET_SOURCE == "roboflow"
          else "owner's own photos (Drive)")
rows = "\n".join(f"| {r['model']} | {r['map50']:.3f} | {r['map50_95']:.3f} | {r['precision']:.3f} | {r['recall']:.3f} | {r['onnx_mb']} | {r['train_minutes']} |" for r in results)
report = f'''# Plate detector: YOLOv8n vs YOLOv8s

Dataset: {source}; images per split {counts}. Scored on the **{EVAL_SPLIT}** split. {EPOCHS} epochs (early stop patience 15), image size {IMAGE_SIZE}, seed 0.
Trained on {gpu or 'CPU'} in Google Colab, {time.strftime('%Y-%m-%d')}.

| Model | mAP50 | mAP50-95 | Precision | Recall | ONNX (MB) | Train (min) |
|---|---|---|---|---|---|---|
{rows}

**Chosen:** {reason}. `plate_detector.onnx` SHA-256 `{sha}`.
CPU latency: run `eval/benchmark_detector.py models/plate_detector.onnx` on the gate machine.
'''
(out / "detector_comparison.md").write_text(report)
(out / "detector_comparison.json").write_text(json.dumps({"dataset": source, "splits": counts, "results": results, "chosen": chosen["model"], "sha256": sha}, indent=2))
print(report)

In [ ]:
from google.colab import files
shutil.make_archive("/content/gatesense_detector_results", "zip", out)
files.download("/content/gatesense_detector_results.zip")

## What to do with the files
The zip has `plate_detector.onnx`, `detector_comparison.md` and `.json`.

1. **Model → GitHub Release** (weights never go in git): on github.com/Akinchan-Jain-7/GateSense → *Releases* → *Draft a new release* → tag `models-v1` → attach `plate_detector.onnx` → *Publish*. Then anyone can run `uv run python -m gatesense_anpr.download_model --sha256 <hash from the report>`.
2. **Results → the repo:** put `detector_comparison.md` and `.json` in `anpr/eval/results/` (or give them to Claude to commit), so the README tables use these real numbers.